# XGBoost regression on caffeine-positive samples + SHAP interpretation

This notebook loads the reduced dataset, keeps only caffeinated specimens (`caffeine_class == 1`), trains an `XGBRegressor`, evaluates performance, and produces SHAP summaries with human-readable feature names (via `column_mapping.csv`).


In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

try:
    # sklearn >= 1.4
    from sklearn.metrics import root_mean_squared_error
    _HAS_RMSE = True
except Exception:
    _HAS_RMSE = False

#coffee_data = pd.read_csv(r"..\\data\\reduced_data_num.csv")
coffee_data = pd.read_csv(r"..\\data\\reduced_data_bin.csv")

column_names = coffee_data.drop(columns=['caffeine_class', 'caffeine_percent']).columns

filtered_data = coffee_data[coffee_data['caffeine_class'] == 1]

X = filtered_data.drop(columns=["caffeine_class", "caffeine_percent"])
y = filtered_data['caffeine_percent']

X.shape, y.shape

FileNotFoundError: [Errno 2] No such file or directory: '..\\\\data\\\\reduced_data_bin.csv'

In [ ]:
from xgboost import XGBRegressor

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

final_model = XGBRegressor(
    objective="reg:squarederror",
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    random_state=42
)
final_model.fit(X_train, y_train)

y_pred = final_model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = root_mean_squared_error(y_test, y_pred) if _HAS_RMSE else mean_squared_error(y_test, y_pred, squared=False)
r2 = r2_score(y_test, y_pred)

print("MAE:", mae)
print("MSE:", mse)
print("RMSE:", rmse)
print("R²:", r2)


## Notes on negative $R^2$

A negative $R^2$ can occur on held-out data (e.g., a test split or cross-validation) when the model's squared prediction error exceeds the variance of the observed response. In that case, the model performs worse than a mean-only baseline on that evaluation set.


In [ ]:
file_path = r'../input/column_mapping.csv'
column_mapping = pd.read_csv(file_path)

name_mapping = dict(zip(column_mapping['new_name'], column_mapping['long_name']))

X_train_renamed = X_train.rename(columns=name_mapping)
X_test_renamed = X_test.rename(columns=name_mapping)

X_train_renamed.head()

In [ ]:
final_model_renamed = XGBRegressor(
    objective="reg:squarederror",
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    random_state=42
)
final_model_renamed.fit(X_train_renamed, y_train)

y_pred2 = final_model_renamed.predict(X_test_renamed)
print("MAE:", mean_absolute_error(y_test, y_pred2))
print("RMSE:", (root_mean_squared_error(y_test, y_pred2) if _HAS_RMSE else mean_squared_error(y_test, y_pred2, squared=False)))
print("R²:", r2_score(y_test, y_pred2))

In [ ]:
import shap
import matplotlib.pyplot as plt

explainer = shap.Explainer(final_model_renamed, X_train_renamed)
shap_values = explainer(X_test_renamed)

shap.summary_plot(shap_values, X_test_renamed, show=False)
plt.tight_layout()
plt.show()

In [ ]:
shap.summary_plot(shap_values, X_test_renamed, plot_type='bar', show=False)
plt.tight_layout()
plt.show()